# 03 | Monthly predictors and coarse-label matching
Aggregate monthly means from a single Sentinel-1 relative orbit and cloud-screened Sentinel-2. First aggregate individual images to 250 m with a minimum valid-subpixel fraction of 0.5; then average available acquisitions. SMAP uses AM retrievals with quality flag exactly 0; at least four usable daily samples per month. The sensors need not observe the same day, a temporal mismatch we disclose.

We align the fine inputs to the actual ingested SMAP grid using area-weighted averaging. A coarse training row requires ≥75% common valid fine area, ≥80% static land fraction, and a quality-screened SMAP target. No SMAP label is copied to 250 m pixels.

## Setup (CPU, standard RAM)
Run this cell first in each notebook. The project uses small monthly raster tiles and tabular ML. If Colab requests a restart, use Runtime → Restart session and rerun from the beginning. Do not change NumPy manually. Files persist in Drive.

In [ ]:
%pip -q install earthengine-api rasterio pyproj scikit-learn joblib
import sys,json
from pathlib import Path
from importlib.metadata import version
for pkg,mod in [('numpy','numpy'),('scikit-learn','sklearn'),('rasterio','rasterio')]:
    loaded=sys.modules.get(mod)
    if loaded is not None and getattr(loaded,'__version__',version(pkg))!=version(pkg):
        raise RuntimeError(f'{pkg} changed. Restart session and rerun.')
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Bagre_SAR_Soil_Moisture')
assert (ROOT/'code'/'sar_workflow.py').is_file(), 'Check Drive folder and shared code.'
sys.path.insert(0,str(ROOT/'code'))
import sar_workflow as sw
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
for sub in ['data','models','outputs','figures','qgis']: (ROOT/sub).mkdir(exist_ok=True)
print('NumPy:',version('numpy'),'scikit-learn:',version('scikit-learn'))

## Connect to Earth Engine
Your existing Cloud project is prefilled. Sign in with the registered Google account when asked. Earth Engine runs the satellite processing remotely.

In [ ]:
import ee_workflow as ew
import ee
PROJECT='YOUR_EARTH_ENGINE_PROJECT_ID'
ew.initialize(PROJECT)

In [ ]:
conf=json.loads((ROOT/'data'/'configuration.json').read_text())
fine,coarse=conf['fine'],conf['coarse']
land=sw.read_tif(ROOT/'data'/'land.tif',fine,['land_fraction','crop_fraction'])

## Download and resume
24 months × three small rasters. Earth Engine computation may take several minutes to much longer depending on availability. Previously completed files are validated and reused. If a request fails, rerun the same cell; it resumes without re-downloading validated rasters. Do not increase resolution mid-run.

In [ ]:
for month in sw.MONTHS:
    print('Processing',month.strftime('%Y-%m'),flush=True)
    ew.acquire_month(ROOT,month,conf)

## Form one row per coarse cell and month
All predictors use exactly the same valid fine pixels. Crop fraction is descriptive, not an input to the retrieval. Record coverage and rejected rows, rather than removing them from the audit. The northeast contiguous quadrant is withheld spatially; April, August and December 2024 (outside the held-out quadrant) are validation; all 2025 is withheld temporally.

In [ ]:
tables=[]
for month in sw.MONTHS:
    key=month.strftime('%Y-%m')
    sar=sw.read_tif(ROOT/'data'/f'sar_{key}.tif',fine,sw.S1)
    optical=sw.read_tif(ROOT/'data'/f'optical_{key}.tif',fine,sw.S2)
    smap=sw.read_tif(ROOT/'data'/f'smap_{key}.tif',coarse,['sm','smap_days'])
    t,_=sw.summarize_month(sar,optical,land,smap,fine,coarse,key); tables.append(t)
all_rows=sw.assign_splits(pd.concat(tables,ignore_index=True))
assert not all_rows.duplicated(['month','row','col']).any()
all_rows.to_csv(ROOT/'data'/'coarse_monthly_table.csv',index=False)
audit=all_rows[all_rows.core].groupby('month').agg(core_cells=('core','size'),predictable=('predictable','sum'),eligible=('eligible','sum'),mean_coverage=('coverage','mean'),median_smap_days=('smap_days','median')).reset_index()
display(audit); audit.to_csv(ROOT/'outputs'/'support_audit.csv',index=False)
display(all_rows[all_rows.eligible].groupby('split').size().rename('eligible_rows'))

In [ ]:
df=all_rows[all_rows.eligible].copy()
df['vv_db']=sw.db(df.vv_power); df['vh_db']=sw.db(df.vh_power)
fig,axes=plt.subplots(1,3,figsize=(12,4),constrained_layout=True)
train=df[df.split=='train']
for ax,name in zip(axes,['vv_db','vh_db','ndvi']):
    h=ax.scatter(train[name],train.sm,c=pd.to_datetime(train.month).dt.month,s=7,alpha=.5,cmap='viridis'); ax.set(xlabel=name,ylabel='SMAP m³/m³')
fig.colorbar(h,ax=axes,label='Month'); fig.savefig(ROOT/'figures'/'03_training_scatter.png',dpi=150); plt.show()

**Checkpoint:** inspect the audit and split counts. Correlation does not establish causation; NDVI and moisture can co-vary seasonally. The scatterplot uses training rows only. If train <100 or validation <25, report insufficient support; do not relax masks to manufacture a result.